# RadImageNet - LSTM GRU RNN Approach

In [1]:
input_monaipath = "/kaggle/input/monai-v060-deep-learning-in-healthcare-imaging/MONAI-1.0.0"
import sys
sys.path.append(input_monaipath)


In [2]:
import monai

2025-09-03 17:36:52.601738: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756921012.776203      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756921012.836673      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import numpy as np

def add_gaussian_noise(img, prob=0.2, mean=0.0, std=0.01):
    if np.random.rand() < prob:
        noise = np.random.normal(mean, std, img.shape).astype(np.float32)
        return np.clip(img + noise, 0.0, 1.0)
    return img

def scale_intensity(img, prob=0.3, factors=0.1):
    if np.random.rand() < prob:
        scale = 1.0 + np.random.uniform(-factors, factors)
        return np.clip(img * scale, 0.0, 1.0)
    return img

def shift_intensity(img, prob=0.3, offsets=0.1):
    if np.random.rand() < prob:
        shift = np.random.uniform(-offsets, offsets)
        return np.clip(img + shift, 0.0, 1.0)
    return img

def augment(img):
    """ img: np.array, already normalized [0,1] """
    img = add_gaussian_noise(img)
    img = scale_intensity(img)
    img = shift_intensity(img)
    return img


# 1.Preprocessing

In [4]:
#  Approach - z index sorting->radimagenet embeddings->gru
# GLOBALS
ALLOWED_TAGS=[
    "SOPClassUID",
    "SOPInstanceUID",
    "Modality",
    "PatientID",
    "SliceThickness",
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "InstanceNumber",
    "ImagePositionPatient",
    "ImageOrientationPatient",
    "FrameOfReferenceUID",
    "SamplesPerPixel",
    "PhotometricInterpretation",
    "Rows",
    "Columns",
    "PixelSpacing",
    "BitsAllocated",
    "BitsStored",
    "HighBit",
    "PixelRepresentation",
    "PixelData"
]
ID_COL = 'SeriesInstanceUID'
LABEL_COLS = [
    'Left Infraclinoid Internal Carotid Artery',
    'Right Infraclinoid Internal Carotid Artery',
    'Left Supraclinoid Internal Carotid Artery',
    'Right Supraclinoid Internal Carotid Artery',
    'Left Middle Cerebral Artery',
    'Right Middle Cerebral Artery',
    'Anterior Communicating Artery',
    'Left Anterior Cerebral Artery',
    'Right Anterior Cerebral Artery',
    'Left Posterior Communicating Artery',
    'Right Posterior Communicating Artery',
    'Basilar Tip',
    'Other Posterior Circulation',
    'Aneurysm Present',
]


In [5]:
# Preprocessing code for one series

# Imports for z index sorting 
import os
import pydicom as dcm
import numpy as np
import cv2
import json

# Radimage net embeddings import
import torch
import torch.nn as nn
import monai
# from monai.transforms import (
#     Compose,RandGaussianNoise, RandScaleIntensity, RandShiftIntensity
# )

# Resize embeddings for lstm 
from torch.utils.data import Dataset

# 1.Load a single scan and return np array for resnet
def load_series(series_path):
    """Load a scan and does z index sorting and with returns npy format"""
    # Load all dicom files in series
    dcm_files = [os.path.join(series_path, f) for f in os.listdir(series_path) if f.endswith(".dcm")]
    datasets = []
    for f in dcm_files:
        try:
            datasets.append(dcm.dcmread(f))
        except:
            continue
            
    # Check metadata availability
    first_ds = datasets[0]
    series_id = getattr(first_ds, "SeriesInstanceUID", "UNKNOWN")

    # Sort  with z index
    try:
        datasets.sort(key=lambda ds: float(ds.ImagePositionPatient[2]))
    except Exception:
        datasets.sort(key=lambda ds: int(ds.InstanceNumber))

    # Extract pixel data
    slices = []
    for ds in datasets:
        arr = ds.pixel_array.astype(np.float32)
        arr = cv2.resize(arr, (224, 224))  # resize for ResNet
        arr = (arr - np.min(arr)) / (np.max(arr) - np.min(arr) + 1e-5)  # normalize
        slices.append(arr)

    volume = np.stack(slices, axis=0)  # shape = (num_slices, 224, 224)

    # Save processed volume
    return volume,series_id

# 2.Rad imagenet embeddings

radimagenet_path = "/kaggle/input/radimagenet_50/pytorch/default/1/ResNet50.pt"
"""
Embedding generator with augmentation using RadImageNet ResNet50
"""
import torch
import torch.nn as nn
from torchvision.models import resnet50


def get_feature_extractor(radimagenet_path: str):
    resnet = monai.networks.nets.resnet50(spatial_dims=2, n_input_channels=3)

    state_dict = torch.load(radimagenet_path, map_location="cpu")
    # checkpoint wrapped like {"state_dict": ...}
    if "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    # remove "backbone." prefix
    new_state_dict = {}
    for k, v in state_dict.items():
        new_k = k.replace("backbone.", "")  # strip prefix
        new_state_dict[new_k] = v

    # load into resnet
    resnet.load_state_dict(new_state_dict, strict=False)

    feature_extractor = nn.Sequential(*list(resnet.children())[:-1])  # remove fc
    feature_extractor.eval()
    return feature_extractor



# Init embedder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Embedder =get_feature_extractor("/kaggle/input/radimagenet_50/pytorch/default/1/ResNet50.pt").to(device)
# Define augmentation pipeline
# augment = Compose([
#     RandGaussianNoise(prob=0.2, mean=0.0, std=0.01),               # scanner noise
#     RandScaleIntensity(factors=0.1, prob=0.3),                     # intensity scaling
#     RandShiftIntensity(offsets=0.1, prob=0.3),                     # intensity shifting
# ])


# 3. Make suitable for lsmtgru model

SEQ_LEN = 800
EMB_DIM = 2048

def embedding_resizer(raw_embedding: np.ndarray, seq_len: int = SEQ_LEN):
    """
    Resize embeddings to fixed length (seq_len, EMB_DIM).
    Also returns an attention mask (1 for real slices, 0 for padding).
    """
    n_slices = raw_embedding.shape[0]

    # Case 1: Too short → pad
    if n_slices < seq_len:
        pad_len = seq_len - n_slices
        pad = np.zeros((pad_len, raw_embedding.shape[1]), dtype=raw_embedding.dtype)
        resized = np.concatenate([raw_embedding, pad], axis=0)
        mask = np.concatenate([np.ones(n_slices), np.zeros(pad_len)])

    # Case 2: Too long → sample evenly
    elif n_slices > seq_len:
        indices = np.linspace(0, n_slices - 1, seq_len).astype(int)
        resized = raw_embedding[indices]
        mask = np.ones(seq_len)

    # Case 3: Exact length
    else:
        resized = raw_embedding
        mask = np.ones(seq_len)

    return resized, mask




In [6]:
"""WARM UP"""

# # Warm up
# # example data series
# series_path='/kaggle/input/rsna-intracranial-aneurysm-detection/series/1.2.826.0.1.3680043.8.498.99887675554378211308175946117895608384'

# # loader is fine
# volume,series_id=load_series(series_path)
# # print(series_id)
# # print(volume.shape)
# # print(np.unique(volume))

# # now embeddings
# slices=volume
# embeddings = []
# with torch.no_grad():
#         for s in slices:
#             # augment slice (still single-channel)
#             s_aug = augment(s[np.newaxis, :, :])  # (1, H, W)

#             # expand to 3 channels (grayscale → RGB-like)
#             img = np.repeat(s_aug, 3, axis=0)  # (3, 224, 224)
#             img = torch.tensor(img, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 3, 224, 224)

#             # forward pass
#             feat = Embedder(img)  # (1, 2048, 1, 1)
#             feat = torch.flatten(feat, 1)  # (1, 2048)

#             embeddings.append(feat.cpu().numpy())

# # stack → (num_slices, 2048)
# embeddings = np.vstack(embeddings)
# # print(f"✅ Generated embeddings for series {series_id} → {embeddings.shape}")

#  # resize
# embedding, mask = embedding_resizer(embeddings, SEQ_LEN)
# embedding = torch.tensor(embedding, dtype=torch.float32)   # (SEQ_LEN, EMB_DIM)
# mask = torch.tensor(mask, dtype=torch.float32)             # (SEQ_LEN,)


'WARM UP'

In [7]:
# print(embedding.shape)
# print(mask.shape)


In [8]:
# # Make predictions
# embedding, mask = embedding.to(device), mask.to(device)
# logits = model(embedding.unsqueeze(0), mask.unsqueeze(0))
# probs = torch.sigmoid(logits).detach().cpu()

In [9]:
# probs.shape

# 2. Model Architecture

In [10]:
# 4. Create and load model

import torch
import torch.nn as nn

class AneurysmGRU(nn.Module):
    def __init__(self,
                 input_dim=2048,
                 hidden_dim=512,
                 num_layers=2,
                 num_classes=14,
                 bidirectional=True,
                 dropout=0.3):
        super(AneurysmGRU, self).__init__()

        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )

        self.bidirectional = bidirectional
        self.hidden_dim = hidden_dim

        # Linear head
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Sequential(
            nn.Linear(out_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, input_dim)
        mask: (batch, seq_len)   [1 = real, 0 = pad]
        """
        # GRU forward
        out, _ = self.gru(x)   # (batch, seq_len, hidden_dim*2)

        if mask is not None:
            mask = mask.unsqueeze(-1)  # (batch, seq_len, 1)
            out = out * mask           # zero out padded timesteps

        # Global average pooling (mask-aware)
        if mask is not None:
            summed = torch.sum(out, dim=1)             # (batch, hidden_dim*2)
            counts = torch.sum(mask, dim=1) + 1e-6     # (batch, 1)
            pooled = summed / counts                   # mean pooling
        else:
            pooled = out.mean(dim=1)

        # Classifier
        logits = self.fc(pooled)  # (batch, num_classes)

        return logits
model = AneurysmGRU(
    input_dim=2048,
    hidden_dim=800,
    num_layers=4,
    num_classes=14,
    bidirectional=True,
    dropout=0.4
)
last_model_path='/kaggle/input/gru_rnn/pytorch/default/1/model_approach_2_fold0_best.pth'
checkpoint = torch.load(last_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
model.eval()

AneurysmGRU(
  (gru): GRU(2048, 800, num_layers=4, batch_first=True, dropout=0.4, bidirectional=True)
  (fc): Sequential(
    (0): Linear(in_features=1600, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=256, out_features=14, bias=True)
  )
)

In [11]:
import polars as pl
import kaggle_evaluation.rsna_inference_server

def _predict_inner(series_path):
    volume,series_id=load_series(series_path)
    # shape = (num_slices, 224, 224)

    slices=volume
    embeddings = []
    with torch.no_grad():
        for s in slices:
            # augment slice (still single-channel)
            s_aug = augment(s)  # (1, H, W)

            # expand to 3 channels (grayscale → RGB-like)
            img = np.repeat(s_aug[np.newaxis, :, :], 3, axis=0)  # (3, 224, 224)
            img = torch.tensor(img, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 3, 224, 224)

            # forward pass
            feat = Embedder(img)  # (1, 2048, 1, 1)
            feat = torch.flatten(feat, 1)  # (1, 2048)

            embeddings.append(feat.cpu().numpy())

    # stack → (num_slices, 2048)
    embeddings = np.vstack(embeddings)

    # resize
    embedding, mask = embedding_resizer(embeddings, SEQ_LEN)
    embedding = torch.tensor(embedding, dtype=torch.float32)   # (SEQ_LEN, EMB_DIM)
    mask = torch.tensor(mask, dtype=torch.float32)             # (SEQ_LEN,)

    # Make predictions
    embedding, mask = embedding.to(device), mask.to(device)
    logits = model(embedding.unsqueeze(0), mask.unsqueeze(0))
    probs = torch.sigmoid(logits).detach().cpu()

    # Now polars dataframe
    predictions_df = pl.DataFrame(
            data=[[series_id] + probs[0].tolist()],
            schema=[ID_COL] + LABEL_COLS,
            orient='row'
        )
    # Return without ID column, as required by the API
    return predictions_df.drop(ID_COL)

In [12]:
sample_ans=_predict_inner('/kaggle/input/rsna-intracranial-aneurysm-detection/series/1.2.826.0.1.3680043.8.498.99887675554378211308175946117895608384')

In [13]:
sample_ans

Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.474285,0.497543,0.527242,0.512829,0.514383,0.525174,0.507274,0.509374,0.508852,0.538578,0.483553,0.489465,0.497759,0.527015


In [14]:
# sample_ans

In [15]:
import sys
import shutil
import warnings
import gc
warnings.filterwarnings('ignore')
def predict(series_path: str) -> pl.DataFrame:
    """
    Top-level prediction function passed to the server.
    It calls the core logic and guarantees cleanup in a `finally` block.
    """
    try:
        # Call the internal prediction logic
        return _predict_inner(series_path)
    except Exception as e:
        #print(f"Error during prediction for {os.path.basename(series_path)}: {e}")
        #print("Using fallback predictions.")
        # Return a fallback dataframe with the correct schema
        conservative_preds = [0.1] * len(LABEL_COLS)
        predictions = pl.DataFrame(
            data=[conservative_preds],
            schema=LABEL_COLS,
            orient='row'
        )
        return predictions
    finally:
        # This code is required to prevent "out of disk space" and "directory not empty" errors.
        # It deletes the shared folder and then immediately recreates it, ensuring it's
        # empty and ready for the next prediction.
        shared_dir = '/kaggle/shared'
        shutil.rmtree(shared_dir, ignore_errors=True)
        os.makedirs(shared_dir, exist_ok=True)
        
        # Also perform memory cleanup here
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

In [16]:
# Main execution

# Initialize the inference server with our main `predict` function.
inference_server = kaggle_evaluation.rsna_inference_server.RSNAInferenceServer(predict)

# Check if the notebook is running in the competition environment or a local session.
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    # make sure to give it an empty directory
    tmp_dir = "/kaggle/working/tmp_gateway"
    os.makedirs(tmp_dir, exist_ok=True)
    inference_server.run_local_gateway(file_share_dir=tmp_dir)
    
    submission_df = pl.read_parquet('/kaggle/working/submission.parquet')
    display(submission_df)



SeriesInstanceUID,Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1.2.826.0.1.3680043.8.498.1005…",0.469454,0.500044,0.56004,0.518191,0.510329,0.559891,0.519575,0.555367,0.518125,0.574766,0.477897,0.470623,0.500769,0.546746
"""1.2.826.0.1.3680043.8.498.1007…",0.424931,0.435053,0.47142,0.487661,0.411473,0.483478,0.455474,0.378516,0.402086,0.415368,0.34628,0.500519,0.431541,0.398371
"""1.2.826.0.1.3680043.8.498.1002…",0.474136,0.497336,0.529902,0.512154,0.513307,0.527894,0.507955,0.517729,0.510522,0.542912,0.485858,0.487234,0.500363,0.530029
